# Plot Y2 Focal Mechanisms — Beach Ball Map
Python equivalent of `I_plotFM_HASH.m`. Reads SKHASH `OUT_Y2/out.txt`, plots beach balls on a map colored by fault type, with station markers and region rectangles.

**Run with FM_ML kernel** (obspy available).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from obspy.imaging.beachball import beach
from pathlib import Path

SKHASH_DIR = Path('/Users/mczhang/Documents/GitHub/SKHASH/SKHASH7')
OUT_FILE   = SKHASH_DIR / 'examples/hash3/OUT_Y2/out.txt'
OUT_DIR    = Path('/Users/mczhang/Documents/GitHub/FM7/02-data/G_FM')
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(OUT_FILE)
print(f'Total FMs: {len(df)}')
print(df['quality'].value_counts().sort_index())

In [ ]:
# ── Fault-type classification from rake (same logic as MATLAB H_read_SKHASH) ──
def classify_fault(rake):
    """N=normal, R=reverse, S=strike-slip, U=undefined."""
    r = rake % 360
    if r > 180: r -= 360   # normalise to -180..180
    if -120 <= r <= -60:   return 'N'
    if   60 <= r <=  120:  return 'R'
    if abs(r) <= 30 or abs(r) >= 150: return 'S'
    return 'U'

FAULT_COLOR = {'N': [0.85, 0.15, 0.15],   # red   — normal
               'R': [0.15, 0.25, 0.85],   # blue  — reverse
               'S': [0.10, 0.70, 0.20],   # green — strike-slip
               'U': [0.40, 0.40, 0.40]}   # grey  — undefined

df['fault_type'] = df['rake'].apply(classify_fault)
print('Fault type distribution:')
print(df['fault_type'].value_counts())

In [ ]:
# ── Map parameters (from I_plotFM_HASH.m) ──────────────────────────────────
LON_LIM = (-130.04, -129.97)
LAT_LIM = (45.908,  46.001)

# Regions
REGIONS = {
    'West': dict(lat=(45.930, 45.953), lon=(-130.029, -130.008)),
    'East': dict(lat=(45.930, 45.971), lon=(-130.0015, -129.975)),
    'ID':   dict(lat=(45.921, 45.929), lon=(-130.004, -129.975)),
}

# Y2 station coordinates (from axial.Y2.sta)
STA_COORDS = {
    'AS1': (45.93356, -129.99920), 'AS2': (45.93377, -130.01410),
    'CC1': (45.95468, -130.00890), 'EC1': (45.94958, -129.97970),
    'EC2': (45.93967, -129.97380), 'EC3': (45.93607, -129.97850),
    'ID1': (45.92573, -129.97800),
    '01B': (46.01935, -130.00526), '02B': (46.00111, -130.02930),
    '03B': (45.99250, -129.99397), '04B': (45.98834, -130.04896),
    '05B': (45.97327, -129.97728), '06B': (45.98178, -130.01423),
    '07B': (45.96950, -130.02963), '08B': (45.96336, -130.00441),
    '09B': (45.97194, -130.05972), '10B': (45.95867, -129.94982),
    '11B': (45.94851, -130.03801), '12B': (45.91473, -130.02173),
    '13B': (45.90472, -129.97240), '14B': (45.89913, -130.00612),
}

# Beach ball size: 0.0005° radius ≈ 55m at Axial latitude
BB_WIDTH   = 0.0012   # degrees (display width in data coords)
BB_SCALE   = 1.3

In [ ]:
def plot_fm_map(events_df, quality_filter, title_suffix, out_png):
    """Plot beach ball map for given quality subset."""
    sub = events_df[events_df['quality'].isin(quality_filter)].copy()
    sub = sub[(sub['origin_lat'] <= LAT_LIM[1]) & (sub['origin_lat'] >= LAT_LIM[0]) &
              (sub['origin_lon'] >= LON_LIM[0]) & (sub['origin_lon'] <= LON_LIM[1])]
    n = len(sub)
    print(f'{out_png.name}: {n} events')

    # Figure aspect: match lon/lat ratio at this latitude
    lon_range = LON_LIM[1] - LON_LIM[0]
    lat_range = LAT_LIM[1] - LAT_LIM[0]
    aspect = (lon_range * np.cos(np.radians(np.mean(LAT_LIM)))) / lat_range
    fig_w = 8.0
    fig_h = fig_w / aspect

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor='white')
    ax.set_facecolor([0.85, 0.92, 0.97])

    # Sort so high-quality events are drawn on top
    qual_order = {'A': 4, 'B': 3, 'C': 2, 'D': 1}
    sub['qrank'] = sub['quality'].map(qual_order)
    sub = sub.sort_values('qrank')

    for _, row in sub.iterrows():
        s, d, r = row['strike'], row['dip'], row['rake']
        color = FAULT_COLOR[row['fault_type']]
        try:
            bb = beach([s, d, r],
                       xy=(row['origin_lon'], row['origin_lat']),
                       width=BB_WIDTH,
                       linewidth=0.3,
                       facecolor=color,
                       alpha=0.85)
            bb.set_transform(ax.transData)
            bb.set_zorder(2)
            ax.add_collection(bb)
        except Exception:
            pass

    # Stations
    sta_lats = [v[0] for v in STA_COORDS.values()]
    sta_lons = [v[1] for v in STA_COORDS.values()]
    ax.plot(sta_lons, sta_lats, 's', color='cyan', markeredgecolor='k',
            markeredgewidth=0.5, markersize=5, zorder=5, label='Stations')

    # Region rectangles
    reg_colors = {'West': 'navy', 'East': 'darkgreen', 'ID': 'purple'}
    for name, reg in REGIONS.items():
        x0, x1 = reg['lon']
        y0, y1 = reg['lat']
        rect = Rectangle((x0, y0), x1-x0, y1-y0,
                          linewidth=1.5, edgecolor=reg_colors[name],
                          facecolor='none', linestyle='--', zorder=6)
        ax.add_patch(rect)
        ax.text(x0, y1 + 0.001, name, fontsize=8,
                color=reg_colors[name], va='bottom')

    # Fault type legend
    legend_handles = [
        mpatches.Patch(color=FAULT_COLOR['N'], label='Normal'),
        mpatches.Patch(color=FAULT_COLOR['R'], label='Reverse'),
        mpatches.Patch(color=FAULT_COLOR['S'], label='Strike-slip'),
        mpatches.Patch(color=FAULT_COLOR['U'], label='Oblique'),
    ]
    ax.legend(handles=legend_handles, loc='lower right', fontsize=8,
              framealpha=0.9, edgecolor='gray')

    ax.set_xlim(LON_LIM)
    ax.set_ylim(LAT_LIM)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    q_str = '+'.join(quality_filter)
    ax.set_title(f'{n} FMs (Y2 Sep 2023–Sep 2024, Quality {q_str})  {title_suffix}',
                 fontsize=11)
    ticks = np.arange(-130.04, -129.96, 0.01)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'{t:.2f}' for t in ticks], fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, linewidth=0.4, color='gray', alpha=0.5)

    plt.tight_layout()
    fig.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {out_png}')

In [ ]:
# ── Plot 1: A+B+C quality (like MATLAB script) ────────────────────────────
plot_fm_map(df, ['A','B','C'],
            'N=normal R=reverse S=strike-slip U=oblique',
            OUT_DIR / 'Y2_FM_map_ABC.png')

# ── Plot 2: A+B only (high confidence) ────────────────────────────────────
plot_fm_map(df, ['A','B'],
            'A+B quality only',
            OUT_DIR / 'Y2_FM_map_AB.png')

print('Done.')

In [ ]:
# ── Plot 3: Depth cross-section — convert lon→km so beach balls stay round ─
def plot_depth_section(events_df, quality_filter, out_png):
    """Depth vs E–W distance (km) cross-section with beach balls."""
    sub = events_df[events_df['quality'].isin(quality_filter)].copy()
    sub = sub[(sub['origin_lat'] <= LAT_LIM[1]) & (sub['origin_lat'] >= LAT_LIM[0]) &
              (sub['origin_lon'] >= LON_LIM[0]) & (sub['origin_lon'] <= LON_LIM[1])]
    n = len(sub)

    # Convert longitude → km east of western limit
    KM_PER_DEG = 111.32 * np.cos(np.radians(np.mean(LAT_LIM)))
    lon_to_km = lambda lon: (lon - LON_LIM[0]) * KM_PER_DEG
    sub['x_km'] = lon_to_km(sub['origin_lon'])
    x_range = lon_to_km(LON_LIM[1])   # ≈ 6 km
    depth_max = 5.0
    BB_KM = 0.12   # beach ball diameter in km — round at equal aspect

    fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
    ax.set_facecolor([0.85, 0.92, 0.97])

    qual_order = {'A': 4, 'B': 3, 'C': 2, 'D': 1}
    sub['qrank'] = sub['quality'].map(qual_order)
    sub = sub.sort_values('qrank')

    for _, row in sub.iterrows():
        s, d, r = row['strike'], row['dip'], row['rake']
        color = FAULT_COLOR[row['fault_type']]
        try:
            bb = beach([s, d, r],
                       xy=(row['x_km'], -row['origin_depth_km']),
                       width=BB_KM,
                       linewidth=0.3,
                       facecolor=color,
                       alpha=0.85)
            bb.set_transform(ax.transData)
            bb.set_zorder(2)
            ax.add_collection(bb)
        except Exception:
            pass

    # x-axis tick labels as longitude
    xtick_lons = np.arange(-130.04, -129.96, 0.01)
    xtick_km   = [lon_to_km(l) for l in xtick_lons]
    ax.set_xticks(xtick_km)
    ax.set_xticklabels([f'{l:.2f}' for l in xtick_lons], fontsize=8)

    ax.set_xlim(0, x_range)
    ax.set_ylim(-depth_max, 0)
    ax.set_yticks(-np.arange(0, depth_max+1, 1))
    ax.set_yticklabels([f'{d:.0f}' for d in np.arange(0, depth_max+1, 1)])
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Depth (km)', fontsize=12)
    q_str = '+'.join(quality_filter)
    ax.set_title(f'{n} FMs — E–W depth section (Y2, Quality {q_str})', fontsize=11)
    ax.grid(True, linewidth=0.4, color='gray', alpha=0.5)

    # Fault-type legend
    legend_handles = [
        mpatches.Patch(color=FAULT_COLOR['N'], label='Normal'),
        mpatches.Patch(color=FAULT_COLOR['R'], label='Reverse'),
        mpatches.Patch(color=FAULT_COLOR['S'], label='Strike-slip'),
        mpatches.Patch(color=FAULT_COLOR['U'], label='Oblique'),
    ]
    ax.legend(handles=legend_handles, loc='lower right', fontsize=8,
              framealpha=0.9, edgecolor='gray')

    plt.tight_layout()
    fig.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_png}')


plot_depth_section(df, ['A','B','C'], OUT_DIR / 'Y2_FM_depth_section_ABC.png')
plot_depth_section(df, ['A','B'],     OUT_DIR / 'Y2_FM_depth_section_AB.png')

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────
print('=== SKHASH Y2 Results Summary ===')
print(f'Total FMs: {len(df)}')
print()
print('Quality:')
print(df['quality'].value_counts().sort_index().to_string())
print()
print('Fault type (all):')
print(df['fault_type'].value_counts().to_string())
print()
print('Fault type (A+B only):')
print(df[df.quality.isin(['A','B'])]['fault_type'].value_counts().to_string())
print()
ab = df[df.quality.isin(['A','B'])]
print(f'A+B mean strike: {ab.strike.mean():.1f}±{ab.strike.std():.1f}')
print(f'A+B mean dip:    {ab.dip.mean():.1f}±{ab.dip.std():.1f}')
print(f'A+B mean rake:   {ab.rake.mean():.1f}±{ab.rake.std():.1f}')